# Implements a random G/G/c queue generator
# Rafael Andrade & Eriky Silva & Frederico Cruz
---

Required packages

In [ ]:
import random as rnd
import heapq as hq
import pandas as pd
import numpy as np
import math

import pickle
from datetime import date

## Queue generator class
---

In [ ]:
class Queue:
    def __init__(self, farr, fdep, tmax, nserv=1, arrbatch=1, depbatch=1):
        self.elements = []
        self.nserv = nserv
        self.farr = farr
        self.fdep = fdep
        self.arrbatch = int(arrbatch)
        self.depbatch = int(depbatch)
        self.tmax = tmax

        self.queue = 0
        self.busyserv = 0
        self.time = 0
        self.events = []

        self.log_time = []
        self.log_event = []
        self.log_queue = []
        self.log_busy = []
        
        self.schedule_arr()


    def schedule_arr(self):
        arr_time = self.time + self.farr()
        hq.heappush(self.events, (arr_time, 'arr'))


    def schedule_dep(self):
        dep_time = self.time + self.fdep()
        hq.heappush(self.events, (dep_time, 'dep'))


    def try_start_service(self):
        while self.busyserv < self.nserv and self.queue > 0:
            batch = min(self.depbatch, self.queue)
            self.queue -= batch
            self.busyserv += 1
            self.schedule_dep()


    def process_arr(self):
        self.schedule_arr()
        self.queue += self.arrbatch
        self.try_start_service()


    def process_dep(self):
        self.busyserv -= 1
        self.try_start_service()


    def log_state(self, event):
        self.log_time.append(self.time)
        self.log_event.append(event)
        self.log_queue.append(self.queue)
        self.log_busy.append(self.busyserv)


    def run(self):
        while self.events and self.time < self.tmax:
            self.time, ev = hq.heappop(self.events)

            if ev == 'arr':
                self.process_arr()
            else:
                self.process_dep()

            self.log_state(ev)

    def print_log(self):
        return pd.DataFrame({
            'time': self.log_time,
            'event': self.log_event,
            'queue': self.log_queue,
            'busy': self.log_busy
        })
    
    def get_metrics(self):
        if len(self.log_time) < 2:
            print("Not enough data")
            return

        total_time = self.log_time[-1]

        area_q = 0.0
        area_b = 0.0
        area_s = 0.0

        for i in range(1, len(self.log_time)):
            dt = self.log_time[i] - self.log_time[i-1]

            q = self.log_queue[i-1]
            b = self.log_busy[i-1]
            s = q + b

            area_q += q * dt
            area_b += b * dt
            area_s += s * dt

        avg_q = area_q / total_time
        avg_b = area_b / total_time
        avg_s = area_s / total_time

        n_arr = sum(1 for e in self.log_event if e == 'arr') * self.arrbatch
        n_dep = sum(1 for e in self.log_event if e == 'dep') * self.depbatch

        utilization = avg_b / self.nserv
        lmbda_eff = n_arr / total_time
        throughput = n_dep / total_time

        w = avg_s / lmbda_eff if lmbda_eff > 0 else 0
        wq = avg_q / lmbda_eff if lmbda_eff > 0 else 0

        return {
        "rho": utilization,
        "L": avg_s,
        "Lq": avg_q,
        "W": w,
        "Wq": wq,
        "avg_in_service": avg_b,
        "throughput": throughput,
        "total_time": total_time,
        "arrivals": n_arr,
        "departures": n_dep
    }

    def print_metrics(self):
        metrics = self.get_metrics()
        print("=== Queue Metrics ===")
        print(f"Rho: {metrics['rho']:.4f}")
        print(f"L: {metrics['L']:.4f}")
        print(f"Lq: {metrics['Lq']:.4f}")
        print(f"W: {metrics['W']:.4f}")
        print(f"Wq: {metrics['Wq']:.4f}")
        print(f"Avg in service: {metrics['avg_in_service']:.4f}")
        print(f"Throughput: {metrics['throughput']:.4f}")
        print(f"Total time: {metrics['total_time']:.4f}")
        print(f"Arrivals: {metrics['arrivals']}")
        print(f"Departures: {metrics['departures']}")


## Simulating queues
---

### Simulating Mx-M-1

#### Monte Carlo functions

In [ ]:
def mc_queue(farr, fdep, tmax, rep_mc=100):
    np.random.seed(2025)
    res = []
    for _ in range(rep_mc):
        q = Queue(farr, fdep, tmax)
        q.run()
        m = q.get_metrics()
        res.append([m['Wq'], m['Lq'], m['rho']])
    
    df = pd.DataFrame(res, columns=['Wq', 'Lq', 'Rho'])
    return df.mean().add_suffix('_mean').to_dict() | df.var().add_suffix('_var').to_dict()


def tab_mc_queue(tmaxs, lambdas, mus, farr, fdep, rep_mc=100):
    tab = []
    total = len(tmaxs) * len(lambdas)
    k = 0
    for lmb, mu in zip(lambdas, mus):
        for tmax in tmaxs:
            k += 1
            print(f"\r{100*k/total:.1f}%", end="")
            stats = mc_queue(farr(lmb), fdep(mu), tmax, rep_mc)
            tab.append({'lambda': lmb, 'mu': mu, 'tmax': tmax} | stats)
    return pd.DataFrame(tab)

#### Simulation

In [ ]:
nbat = 5
tmaxs = [100, 500, 1000, 5000, 10000]
rhos = np.arange(0.5, 0.9 + 0.1, 0.1)
lambdas = np.array([0.5, 1.0, 1.5])
mus = np.array([2.0, 2.0, 2.0])
rep_mc = 100

sim_MxM1_const = tab_mc_queue(
    tmaxs, lambdas/nbat, mus,
    farr = lambda lmbda: lambda: np.random.exponential(1/lmbda),
    fdep = lambda mu: lambda: np.random.exponential(1/mu),
    rep_mc = rep_mc
)

In [ ]:
sim_MxM1_const.head(10)

### Simulating M-Mx-1

### Save results and clear variables

In [ ]:
filename = f"sim_{date.today()}.pkl"

results = {
    'rhos': rhos, 'lambdas': lambdas, 'mus': mus, 'nbat': nbat,
    'tmaxs': tmaxs, 'rep_mc': rep_mc,
    'sim_MxM1_const': sim_MxM1_const
    'sim_MMx1_const': sim_MMx1_const,
}

with open(filename, 'wb') as f:
    pickle.dump(results, f)

del rhos, lambdas, mus, tmaxs, rep_mc, nbat, filename, results

## Testing queue generator
---

### M-M-1 test

Theoretical metrics

In [ ]:
def mmc_metrics(lmbda, mu, c):
    rho = lmbda / (c * mu)
    s   = sum((lmbda / mu)**n / math.factorial(n) for n in range(c))
    t   = (lmbda / mu)**c / (math.factorial(c) * (1 - rho))
    P0  = 1 / (s + t)
    Pw  = t * P0
    Wq  = Pw / (c * mu - lmbda)
    W   = Wq + 1 / mu
    Lq  = lmbda * Wq
    L   = lmbda * W
    return rho, Pw, Wq, W, Lq, L

In [ ]:
nserv  = 3
rho    = 0.5
lmbda  = 2.0
mu     = lmbda / (nserv * rho)

rho, Pw, Wq, W, Lq, L = mmc_metrics(lmbda, mu, nserv)

print("Rho:", round(rho, 4))
print("W  :", round(W, 4))
print("Wq :", round(Wq, 4))
print("Lq :", round(Lq, 4))
print("L  :", round(L, 4))
print("Pw :", round(Pw, 4))

Simulated metrics

In [ ]:
farr = lambda: rnd.expovariate(lmbda)
fdep = lambda: rnd.expovariate(mu)

q = Queue(nserv=nserv, farr=farr, fdep=fdep, arrbatch=1, depbatch=1, tmax=10000)

q.run()
q.print_metrics()

### Mx-M-1 test

Theoretical metrics

In [ ]:
def mkm1_metrics(lmbda_event, batch_size, mu):
    lmbda_eff = lmbda_event * batch_size
    rho       = lmbda_eff / mu
    EX2       = batch_size**2
    L         = (rho + (lmbda_event / mu) * EX2) / (2 * (1 - rho))
    W         = L / lmbda_eff
    Wq        = W - 1 / mu
    Lq        = lmbda_eff * Wq
    return lmbda_eff, rho, L, Lq, W, Wq

In [ ]:
lmbda_val = 0.2
k_val     = 5
mu_val    = 2.0

lmbda_eff, rho, L, Lq, W, Wq = mkm1_metrics(lmbda_val, k_val, mu_val)

print("rho       :", round(rho, 4))
print("L         :", round(L, 4))
print("Lq        :", round(Lq, 4))
print("W         :", round(W, 4))
print("Wq        :", round(Wq, 4))
print("lambda_eff:", round(lmbda_eff, 4))

Simulated metrics

In [ ]:
farr = lambda: rnd.expovariate(lmbda_val)
fdep = lambda: rnd.expovariate(mu_val)

q = Queue(nserv=1, farr=farr, fdep=fdep, arrbatch=k_val, depbatch=1, tmax=100000)

q.run()
q.print_metrics()

### M-G-1 test

Theoretical metrics

In [ ]:
def mg1_metrics(lmbda, ES, VarS):
    rho = lmbda * ES
    Cv2 = VarS / (ES**2)

    Lq  = ((1 + Cv2) / 2) * (rho**2 / (1 - rho))
    W   = ((1 + Cv2) / 2) * (rho / (1/ES - lmbda)) + ES
    L   = Lq + rho
    Wq  = W - ES

    return rho, L, Lq, W, Wq

In [ ]:
lmbda = 0.8
a, b  = 0.5, 1.5

ES    = (a + b) / 2
VarS  = (b - a)**2 / 12

rho, L, Lq, W, Wq = mg1_metrics(lmbda, ES, VarS)

print("rho:", round(rho, 4))
print("L  :", round(L, 4))
print("Lq :", round(Lq, 4))
print("W  :", round(W, 4))
print("Wq :", round(Wq, 4))

Simulated metrics

In [ ]:
q = Queue(
    farr = lambda: rnd.expovariate(lmbda),
    fdep = lambda: rnd.uniform(a, b),
    tmax = 200000
)

q.run()
q.print_metrics()